# M5 — Unsupervised YouTube lexical-variant mining

M5 uses **unpaired YouTube comments only** to discover likely informal→formal Wolof token mappings. It does not read manual normalization targets. Accepted mappings are reversed and applied to the same 17,777 formal sources used by M0/M1.

A run made before the gold split is locked is explicitly **provisional**. The final run must exclude every comment from development and test videos and regenerate all mappings and pairs.

## Step 1 — Environment and imports

The mining stage is CPU-intensive but requires no model download and no GPU. `rapidfuzz`, pandas, and a Parquet engine are required.

In [1]:
# %pip install -U pandas pyarrow rapidfuzz

import json
from pathlib import Path
import sys
import time

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLEAN_COMMENTS_PATH, FRENCH_WORDLIST_PATH, GOLD_DEV_PATH,
    GOLD_SPLIT_MANIFEST_PATH, GOLD_TEST_PATH, M5_ACCEPTED_MAPPINGS_PATH,
    M5_GENERATION_REVIEW_PATH, M5_MAPPING_REVIEW_PATH, M5_MAPPINGS_PATH,
    M5_YOUTUBE_MANIFEST_PATH, M5_YOUTUBE_PATH, TRAIN_PARQUET_PATH,
    WOLOF_VOCAB_PATH,
)
from src.generation.benchmark_baselines import load_formal_sources
from src.generation.youtube_variant_mining import (
    MiningConfig, build_m5_pairs, finalize_m5_manifest, likely_capitalized_tokens,
    mine_variant_mappings, word_tokens,
    write_m5_outputs,
)

## Step 2 — Frozen mining configuration

These initial thresholds are method parameters, not learned gold labels. Do not tune them against the gold test set. A small `MAX_COMMENTS` is useful for debugging but always makes the output provisional.

In [2]:
CONFIG = MiningConfig(
    min_frequency=5,
    min_token_length=3,
    max_token_length=30,
    context_window=2,
    candidate_pool_limit=200,
    acceptance_score=0.68,
    minimum_margin=0.03,
    max_edits_per_sentence=2,
    seed=2026,
)
MAX_COMMENTS = None  # Example debug value: 20_000; use None for the full run.
ALLOW_PROVISIONAL = False  # Gold is locked: never permit a provisional final export.
OVERWRITE_OUTPUTS = True  # Set False when the final post-split artifacts are frozen.
REVIEW_SIZE = 100
FINALIZE_M5_AFTER_REVIEW = False
CONFIG

MiningConfig(min_frequency=5, min_token_length=3, max_token_length=30, context_window=2, max_context_items=50, candidate_pool_limit=200, max_length_difference=3, minimum_char_similarity=0.5, acceptance_score=0.68, minimum_margin=0.03, character_weight=0.4, rule_weight=0.3, context_weight=0.2, frequency_weight=0.1, max_edits_per_sentence=2, capitalization_min_count=3, capitalization_ratio=0.6, seed=2026)

## Step 3 — Determine leakage-safe video eligibility

When the locked split exists, all development/test video URLs are excluded before token counting. Before that point, the notebook may run only as a provisional engineering experiment.

In [3]:
split_is_locked = (
    GOLD_SPLIT_MANIFEST_PATH.exists()
    and GOLD_DEV_PATH.exists()
    and GOLD_TEST_PATH.exists()
)
excluded_videos = set()
if split_is_locked:
    for split_path in (GOLD_DEV_PATH, GOLD_TEST_PATH):
        split = pd.read_csv(split_path)
        if 'video_url' not in split.columns:
            raise ValueError(f'{split_path} has no video_url column')
        excluded_videos.update(split.video_url.dropna().astype(str).str.strip())

provisional = (not split_is_locked) or (MAX_COMMENTS is not None)
if provisional and not ALLOW_PROVISIONAL:
    raise RuntimeError('M5 cannot be finalized before locked dev/test videos are excluded.')
print(f'Locked split available: {split_is_locked}')
print(f'Excluded dev/test videos: {len(excluded_videos):,}')
print(f'Run status: {"PROVISIONAL" if provisional else "FINAL-ELIGIBLE"}')

Locked split available: True
Excluded dev/test videos: 2
Run status: FINAL-ELIGIBLE


## Step 4 — Load eligible unpaired comments and formal resources

In [4]:
comments = pd.read_csv(
    CLEAN_COMMENTS_PATH, usecols=['text', 'clean_comment', 'video_url']
)
comments['video_url'] = comments.video_url.fillna('').astype(str).str.strip()
comments = comments.loc[~comments.video_url.isin(excluded_videos)].copy()
comments = comments.loc[comments.clean_comment.fillna('').str.strip().ne('')].copy()
if MAX_COMMENTS is not None:
    comments = comments.head(MAX_COMMENTS).copy()

formal_sources = load_formal_sources(TRAIN_PARQUET_PATH)
parallel = pd.read_parquet(TRAIN_PARQUET_PATH, columns=['french'])
formal_vocab = {
    line.strip().casefold() for line in WOLOF_VOCAB_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
}
french_vocab = set()
for sentence in parallel.french.fillna(''):
    french_vocab.update(word_tokens(sentence))
if FRENCH_WORDLIST_PATH.exists():
    lexique = pd.read_csv(FRENCH_WORDLIST_PATH, sep='\t', low_memory=False)
    word_column = '1_Mot' if '1_Mot' in lexique.columns else lexique.columns[0]
    french_vocab.update(lexique[word_column].dropna().astype(str).str.casefold().str.strip())
likely_entities = likely_capitalized_tokens(
    comments.text.fillna(''),
    min_count=CONFIG.capitalization_min_count,
    minimum_ratio=CONFIG.capitalization_ratio,
)
formal_entity_candidates = likely_capitalized_tokens(
    formal_sources.formal_wolof, min_count=1, minimum_ratio=1.0,
    ignore_first_token=True,
)
likely_entities.update(formal_entity_candidates)

print(f'Eligible comments: {len(comments):,} from {comments.video_url.nunique():,} videos')
print(f'Formal sources:    {len(formal_sources):,}')
print(f'Formal vocabulary: {len(formal_vocab):,}')
print(f'French guard list: {len(french_vocab):,}')
print(f'Capitalization-protected tokens: {len(likely_entities):,}')

Eligible comments: 239,795 from 191 videos
Formal sources:    17,777
Formal vocabulary: 32,178
French guard list: 181,019
Capitalization-protected tokens: 7,753


## Step 5 — Mine and score informal→formal mappings

For each frequent YouTube OOV token, the miner retrieves formal candidates through Wolof-oriented canonical signatures and character n-grams. Its score combines character similarity (0.40), rule/signature compatibility (0.30), local-context overlap (0.20), and frequency evidence (0.10). French-list matches, low scores, and ambiguous margins are rejected.

This is the longest cell. On the full corpus it may take several minutes depending on CPU and vocabulary size.

In [5]:
started = time.perf_counter()
candidates = mine_variant_mappings(
    comments.clean_comment.fillna('').tolist(),
    formal_sources.formal_wolof.tolist(),
    formal_vocab,
    french_vocab,
    protected_tokens=likely_entities,
    config=CONFIG,
)
elapsed = time.perf_counter() - started
accepted = candidates.loc[candidates.status.eq('accepted')].copy()
print(f'Mining time:       {elapsed / 60:.2f} minutes')
print(f'OOV candidates:    {len(candidates):,}')
print(f'Accepted mappings: {len(accepted):,}')
print(f'Acceptance rate:   {len(accepted) / max(1, len(candidates)):.1%}')

Mining time:       0.83 minutes
OOV candidates:    19,828
Accepted mappings: 114
Acceptance rate:   0.6%


## Step 6 — Inspect mapping evidence and rejection reasons

In [6]:
display(candidates.reason.value_counts(dropna=False))
display(accepted[[
    'informal_token', 'formal_token', 'frequency', 'score', 'margin',
    'char_similarity', 'rule_similarity', 'context_similarity',
]].head(30))
display(candidates.loc[candidates.reason.eq('ambiguous_margin')].head(20))

reason
low_score                          10476
likely_french                       7463
likely_entity_by_capitalization      748
low_character_similarity             559
too_short                            196
excessive_repetition                 181
accepted                             114
no_candidate                          68
ambiguous_margin                      23
Name: count, dtype: int64

,informal_token,formal_token,frequency,score,margin,char_similarity,rule_similarity,context_similarity
119,kine,kinne,1402,0.687797,0.094921,0.800000,1.0,0.000000
131,meune,menne,1258,0.686784,0.053107,0.800000,1.0,0.000000
140,moma,mooma,1153,0.686702,0.095418,0.800000,1.0,0.003665
174,nane,naane,959,0.684722,0.053808,0.800000,1.0,0.002373
206,nonou,nonu,798,0.683505,0.080256,0.800000,1.0,0.004878
219,lalale,lalalee,770,0.705053,0.042857,0.857143,1.0,0.000000
248,yayam,yaayam,689,0.696608,0.060805,0.833333,1.0,0.010587
260,kouko,kuko,665,0.682311,0.141266,0.800000,1.0,0.007423
268,ndeysane,ndeysaane,652,0.716894,0.130905,0.888889,1.0,0.003484
277,topou,toppu,636,0.681813,0.114557,0.800000,1.0,0.007018


,informal_token,frequency,formal_token,score,second_score,margin,char_similarity,rule_similarity,context_similarity,frequency_score,candidate_count,status,reason
37,mome,2898,moome,0.694587,0.694587,0.000000,0.800000,1.0,0.000000,0.745870,200,rejected,ambiguous_margin
263,adouna,657,aduna,0.707701,0.698919,0.008782,0.833333,1.0,0.068273,0.607129,200,rejected,ambiguous_margin
289,yéne,617,yéene,0.682814,0.680368,0.002445,0.800000,1.0,0.013438,0.601261,200,rejected,ambiguous_margin
318,yakar,568,yaakar,0.694352,0.683608,0.010743,0.833333,1.0,0.008326,0.593532,200,rejected,ambiguous_margin
432,yene,440,yéene,0.680623,0.680388,0.000236,0.800000,1.0,0.018272,0.569690,200,rejected,ambiguous_margin
570,louma,333,luma,0.681896,0.677310,0.004586,0.800000,1.0,0.037634,0.543689,200,rejected,ambiguous_margin
601,billay,315,billaay,0.696708,0.687184,0.009524,0.857143,1.0,0.000000,0.538506,200,rejected,ambiguous_margin
668,geune,289,genne,0.682017,0.673047,0.008970,0.800000,1.0,0.044850,0.530473,200,rejected,ambiguous_margin
692,setan,281,seetan,0.686526,0.686119,0.000407,0.833333,1.0,0.002037,0.527856,200,rejected,ambiguous_margin
813,takal,241,taakal,0.685521,0.684688,0.000833,0.833333,1.0,0.004167,0.513544,200,rejected,ambiguous_margin


## Step 7 — Generate M5 synthetic sentences

Accepted mappings are inverted. Each formal sentence receives at most two deterministic, frequency/score-weighted lexical replacements. No general Beqi rule is applied here; that later combination belongs to M6. Punctuation and spacing are retained.

In [7]:
m5 = build_m5_pairs(
    formal_sources, accepted, config=CONFIG, provisional=provisional
)
assert len(m5) == len(formal_sources)
assert m5.source_id.tolist() == formal_sources.source_id.tolist()
assert m5.formal_wolof.notna().all() and m5.informal_wolof.notna().all()

print(f'Generated rows: {len(m5):,}')
print(f'Changed rows:   {m5.changed.sum():,} ({m5.changed.mean():.1%})')
print(f'Unchanged rows: {(~m5.changed).sum():,}')
display(m5.loc[m5.changed, [
    'formal_wolof', 'informal_wolof', 'edit_count', 'applied_rules'
]].sample(min(20, int(m5.changed.sum())), random_state=CONFIG.seed))

Generated rows: 17,777
Changed rows:   1,821 (10.2%)
Unchanged rows: 15,956


,formal_wolof,informal_wolof,edit_count,applied_rules
17446,Cër bi gëna am solo ci liy dundal yaram mooy x...,Cër bi gëna am solo ci liy dundal yaram mooy x...,1,"[{""position"": 10, ""formal"": ""xol"", ""informal"":..."
14512,"Yeglekaay yi neenañu Jones defa ""mer"" bi xale ...","Yeglekaay yi neenañu Jones defa ""mer"" bi xale ...",2,"[{""position"": 9, ""formal"": ""jigeen"", ""informal..."
15412,Sunu baykat yu am jom yi war nañu leen taxawoo...,Sunu baykat yu am jom yi war nañu leen taxawoo...,1,"[{""position"": 22, ""formal"": ""waroon"", ""informa..."
13062,"Ba kël ga agsee ci Suuf sit e dug ndéneer gi, ...","Ba kël ga agsee ci Suuf sit e dug ndéneer gi, ...",1,"[{""position"": 16, ""formal"": ""suba"", ""informal""..."
10596,"Jàppalewul bu baax coro liise bimu am, ba nopp...","Jàppalewul bu baax coro liise bimou am, ba nop...",1,"[{""position"": 5, ""formal"": ""bimu"", ""informal"":..."
6050,"Waaye, fàww nga xam ni ñuy toppatoo dëru liege...","Waaye, fàww nga xam ni ñuy toppatoo dëru liege...",2,"[{""position"": 9, ""formal"": ""suko"", ""informal"":..."
12023,Ci suba teel gi laa seetloo ne Booy Jinne rëcc...,Ci souba teel gi laa seetloo ne Booy Jinne rëc...,1,"[{""position"": 1, ""formal"": ""suba"", ""informal"":..."
12370,Lu gënsaa neew 384 nit ñakkna ñu seen bakkan c...,Lu gënsaa neew 384 nit ñakkna ñu seen bakkan c...,1,"[{""position"": 32, ""formal"": ""limu"", ""informal""..."
2851,"Ci kanamu takk-der yi, ki ñu doon tuumaal nang...","Ci kanamou takk-der yi, ki ñu doon tuumaal nan...",1,"[{""position"": 1, ""formal"": ""kanamu"", ""informal..."
17680,"Waaye, jigeen ñi am nañu lu mu néew néew ndam ...","Waaye, djigeen ñi am nañu lu mu néew néew ndam...",2,"[{""position"": 1, ""formal"": ""jigeen"", ""informal..."


## Step 8 — Export candidates, accepted mappings, M5 pairs, reviews, and manifest

The manifest records whether this run is provisional. Do not use a provisional M5 file in the final benchmark.

In [8]:
output_paths = [
    M5_MAPPINGS_PATH, M5_ACCEPTED_MAPPINGS_PATH, M5_MAPPING_REVIEW_PATH,
    M5_GENERATION_REVIEW_PATH, M5_YOUTUBE_PATH, M5_YOUTUBE_MANIFEST_PATH,
]
if not OVERWRITE_OUTPUTS and any(path.exists() for path in output_paths):
    raise FileExistsError('M5 outputs already exist. Enable overwrite intentionally.')

manifest = write_m5_outputs(
    candidates, m5, candidates_path=M5_MAPPINGS_PATH,
    accepted_path=M5_ACCEPTED_MAPPINGS_PATH,
    mapping_review_path=M5_MAPPING_REVIEW_PATH,
    generation_review_path=M5_GENERATION_REVIEW_PATH,
    pairs_path=M5_YOUTUBE_PATH, manifest_path=M5_YOUTUBE_MANIFEST_PATH,
    comments_path=CLEAN_COMMENTS_PATH, formal_path=TRAIN_PARQUET_PATH,
    vocabulary_path=WOLOF_VOCAB_PATH, excluded_videos=excluded_videos,
    eligible_comment_rows=len(comments), provisional=provisional,
    config=CONFIG, review_size=REVIEW_SIZE,
)
manifest

{'method': 'm5_youtube_mined_variants',
 'created_at_utc': '2026-08-08T14:42:41.822169+00:00',
 'provisional': False,
 'leakage_status': 'gold dev/test videos excluded',
 'comments_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\youtube\\clean_data.csv',
 'comments_sha256': '800b16ade3e9b862ba813baf3ae2c3f4fd1d973f21ec18aa591eddebb69d2e47',
 'formal_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\translation data\\train-00000-of-00001.parquet',
 'formal_sha256': 'bd4910104fc121ff6924ca9a8b1a0d6bdaa6f28601108703cc9b29b2dc9eb27e',
 'vocabulary_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\vocabulary.txt',
 'vocabulary_sha256': '52098f61c7f0f5444e7cd6c10f6a68db8a530d74e46e0d589bc614f9c9e31389',
 'eligible_comment_rows': 239795,
 'excluded_video_count': 2,
 'candidate_mappings': 19828,
 'accepted_mappings': 114,
 'generated_rows': 17777,
 'changed_rows': 1821,
 'unchanged_rows': 15956,
 'configuration': {'min_frequency': 5,
  'min_token_length': 3,
  'max_token_length':

## Step 9 — Required manual work

1. Review `artifacts/m5_mapping_review_sample.csv`: decide whether each mined informal→formal pair is correct.
2. Review `data/synthetic/m5_generation_review_sample.csv`: assess meaning preservation and whether the result resembles plausible informal Wolof.
3. Report reviewed mapping precision and generation-quality distributions. Do not consult gold test targets.
4. Inspect likely French words and named entities among false mappings. Clean comments are lowercased, so automatic entity protection is necessarily limited.
5. If thresholds are revised after review, record a new method/configuration version and freeze it before final test evaluation.
6. After the gold split is locked, rerun the full notebook with `MAX_COMMENTS=None`; verify `provisional: false` in the manifest.

## Interpretation limitation

M5 generation quality is limited by lexical coverage: formal tokens without a reviewed mined variant remain unchanged. Keep those rows in the common source export and report the unchanged rate. A later size-matched training condition may sample changed rows, but that filtering rule must be applied and documented consistently.

In [ ]:
# Run this only after filling both 100-row review CSVs. It does not regenerate M5.
if FINALIZE_M5_AFTER_REVIEW:
    finalized_m5_manifest = finalize_m5_manifest(
        M5_YOUTUBE_MANIFEST_PATH, M5_MAPPING_REVIEW_PATH,
        M5_GENERATION_REVIEW_PATH,
    )
    finalized_m5_manifest
else:
    print('M5 review finalization is disabled until both review sheets are complete.')